<a href="https://colab.research.google.com/github/egor210709/MY_FIRST_TEST_REP/blob/main/%D0%BA%D0%BE%D0%B4_%D0%BF%D1%80%D0%BE%D0%B3%D1%80%D0%B0%D0%BC%D0%BC%D1%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Установка необходимых библиотек (в Colab обычно уже есть, но на всякий случай)
!pip install plotly openpyxl

# 2. Импорт библиотек
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from google.colab import files  # для загрузки/скачивания файлов

# 3. Загрузка данных из Excel-файла
#    Для Colab: загрузите файл Solar_Taj.xlsx через интерфейс (значок папки -> загрузить)
#    или смонтируйте Google Диск. Здесь используем простой upload.
print("Пожалуйста, загрузите файл Solar_Taj.xlsx")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Читаем Excel, пропуская возможные служебные строки (если есть)
# По данным из описания, столбцы: A=YYYY-MM-DD, B=HH:MM (LST), C=Zenith (deg), D=ETR (Wh/m^2)
df_raw = pd.read_excel(filename, header=0, names=['Date', 'Time', 'Zenith', 'ETR'])

print("Первые 5 строк исходных данных:")
print(df_raw.head())
print("\nПоследние 5 строк исходных данных:")
print(df_raw.tail())

# 4. Преобразование столбцов Date и Time в единый datetime
#    В столбце Date встречаются значения вида "2010-01-01 00:00:00" – возьмём только дату.
#    Столбец Time содержит строки "01:00:00" ... "1 day, 0:00:00" (последнее – 24:00).
#    Нужно аккуратно преобразовать время.

def clean_time(t):
    if isinstance(t, str) and 'day' in t:
        return '24:00:00'
    return t

df_raw['Time_clean'] = df_raw['Time'].astype(str).apply(clean_time)
df_raw['Date_only'] = pd.to_datetime(df_raw['Date']).dt.date
df_raw['DateTime'] = pd.to_datetime(df_raw['Date_only'].astype(str) + ' ' + df_raw['Time_clean'], errors='coerce')

# Удалим строки с некорректным временем (если есть)
df = df_raw.dropna(subset=['DateTime']).copy()
df.set_index('DateTime', inplace=True)
df = df[['Zenith', 'ETR']]

print("\nПосле обработки даты/времени:")
print(df.head())

# 5. График всего временного ряда (интерактивный)
fig_full = px.line(df, y='ETR', title='Полный временной ряд ETR (Wh/m²)',
                   labels={'index': 'Дата и время', 'value': 'ETR, Wh/m²'})
fig_full.show()

# 6. Фильтр только за январь 2010
jan_2010 = df.loc['2010-01-01':'2010-01-31 23:59:59'].copy()
print(f"\nДанные за январь 2010: {len(jan_2010)} записей")
print(jan_2010.head())

# 7. Группировка по дням: среднесуточные и максимальные значения ETR
daily_stats = jan_2010.resample('D').agg(
    Среднее_ETR=('ETR', 'mean'),
    Максимум_ETR=('ETR', 'max')
).dropna()

print("\nАгрегированные данные по дням")
print(daily_stats.head())



# 8. Интерактивные графики для января
fig_daily = make_subplots(rows=2, cols=1, shared_xaxes=True,
                          subplot_titles=('Среднесуточная инсоляция', 'Максимальная инсоляция за сутки'))

fig_daily.add_trace(go.Scatter(x=daily_stats.index, y=daily_stats['Среднее_ETR'],
                               mode='lines+markers', name='Средняя'), row=1, col=1)
fig_daily.add_trace(go.Scatter(x=daily_stats.index, y=daily_stats['Максимум_ETR'],
                               mode='lines+markers', name='Максимум'), row=2, col=1)

fig_daily.update_layout(height=600, title_text="Статистика инсоляции за январь 2010",
                        xaxis_title="Дата", yaxis_title="ETR, Wh/m²")
fig_daily.show()

# 9. Сохранение графиков как изображений (PNG)
#    plotly позволяет сохранять статические изображения, но нужен kaleido.
!pip install -q kaleido
fig_full.write_image("full_series.png")
fig_daily.write_image("daily_stats_jan.png")
print("Изображения сохранены: full_series.png, daily_stats_jan.png")

# 10. Сохранение агрегированных данных в Excel (XLSX)
daily_stats.to_excel("daily_stats_jan.xlsx", sheet_name="Январь_2010")
print("Файл daily_stats_jan.xlsx создан")

# 11. Скачивание всех результатов на локальный компьютер
files.download("full_series.png")
files.download("daily_stats_jan.png")
files.download("daily_stats_jan.xlsx")


Пожалуйста, загрузите файл Solar_Taj.xlsx


Saving Solar_Taj (1) (2) (1).xlsx to Solar_Taj (1) (2) (1) (2).xlsx
Первые 5 строк исходных данных:
        Date      Time Zenith  ETR
0 2010-01-01  01:00:00  99.0     0
1 2010-01-01  02:00:00  99.0     0
2 2010-01-01  03:00:00  99.0     0
3 2010-01-01  04:00:00  99.0     0
4 2010-01-01  05:00:00  99.0     0

Последние 5 строк исходных данных:
           Date            Time Zenith  ETR
8755 2010-12-31        20:00:00  99.0     0
8756 2010-12-31        21:00:00  99.0     0
8757 2010-12-31        22:00:00  99.0     0
8758 2010-12-31        23:00:00  99.0     0
8759 2010-12-31  1 day, 0:00:00  99.0     0

После обработки даты/времени:
                    Zenith  ETR
DateTime                       
2010-01-01 01:00:00  99.0     0
2010-01-01 02:00:00  99.0     0
2010-01-01 03:00:00  99.0     0
2010-01-01 04:00:00  99.0     0
2010-01-01 05:00:00  99.0     0



Данные за январь 2010: 713 записей
                    Zenith  ETR
DateTime                       
2010-01-01 01:00:00  99.0     0
2010-01-01 02:00:00  99.0     0
2010-01-01 03:00:00  99.0     0
2010-01-01 04:00:00  99.0     0
2010-01-01 05:00:00  99.0     0

Агрегированные данные по дням
            Среднее_ETR  Максимум_ETR
DateTime                             
2010-01-01   183.000000           683
2010-01-02   183.739130           685
2010-01-03   184.521739           687
2010-01-04   185.434783           690
2010-01-05   186.217391           692


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
